# 第7回講義 宿題

### 課題
RNNを用いてIMDbのsentiment analysisを実装してみましょう．

ネットワークの形などに制限はとくになく，今回のLessonで扱った内容以外の工夫も組み込んでもらって構いません．

### 目標値
F値：0.85

### ルール

- **以下のセルで指定されている`x_train`, `t_train`以外の学習データは使わないでください．**

### 提出方法

- 2つのファイルを提出していただきます．
    1. テストデータ (`x_test`) に対する予測ラベルをcsv形式で保存し，**Omnicampusの宿題タブから「第7回 回帰結合型ニューラルネットワーク」を選択して**提出してください．
    2. それに対応するpythonのコードを　ファイル＞ダウンロード＞.pyをダウンロード　から保存し，**Omnicampusの宿題タブから「第7回 回帰結合型ニューラルネットワーク (code)」を選択して**提出してください．pythonファイル自体の提出ではなく，「提出内容」の部分にコード全体をコピー&ペーストしてください．

- なお，採点は1で行い，2はコードの確認用として利用します（成績優秀者はコード内容を公開させていただくかもしれません）．コードの内容を変更した場合は，**1と2の両方を提出し直してください**．


### 評価方法

- 予測ラベルの`t_test`に対するF値で評価します．
- 即時採点しLeader Boardを更新します．（採点スケジュールは別アナウンス）
- 締切時の点数を最終的な評価とします．



### ドライブのマウント

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 作業ディレクトリを指定
work_dir = 'drive/MyDrive/Colab Notebooks/DLBasics2025_colab'

### データの読み込み（このセルは修正しないでください）

In [4]:
!pip install portalocker

import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from collections import Counter
import pandas as pd
import string
import re
from typing import List, Union

seed = 1234
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)


# 学習データ
x_train = np.load(work_dir + '/Lecture07/data/x_train.npy', allow_pickle=True)
t_train = np.load(work_dir + '/Lecture07/data/t_train.npy', allow_pickle=True)

# 検証データを取る
x_train, x_valid, t_train, t_valid = train_test_split(x_train, t_train, test_size=0.2, random_state=seed)

# テストデータ
x_test = np.load(work_dir + '/Lecture07/data/x_test.npy', allow_pickle=True)


def text_transform(text: List[int], max_length=256):
    # <BOS>はすでに1で入っている．<EOS>は2とする．
    text = text[:max_length - 1] + [2]

    return text, len(text)

def collate_batch(batch):
    label_list, text_list, len_seq_list = [], [], []

    for sample in batch:
        if isinstance(sample, tuple):
            label, text = sample

            label_list.append(label)
        else:
            text = sample.copy()

        text, len_seq = text_transform(text)
        text_list.append(torch.tensor(text))
        len_seq_list.append(len_seq)

    # NOTE: 宿題用データセットでは<PAD>は3です．
    return torch.tensor(label_list), pad_sequence(text_list, padding_value=3).T, torch.tensor(len_seq_list)


word_num = np.concatenate(np.concatenate((x_train, x_test))).max() + 1
print(f"単語種数: {word_num}")

単語種数: 88587


### 実装

In [9]:
batch_size = 128

train_dataloader = DataLoader(
    [(t, x) for t, x in zip(t_train, x_train)],
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_batch,
)
valid_dataloader = DataLoader(
    [(t, x) for t, x in zip(t_valid, x_valid)],
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_batch,
)
test_dataloader = DataLoader(
    x_test,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_batch,
)

In [10]:
def torch_log(x):
    return torch.log(torch.clamp(x, min=1e-10))


class Embedding(nn.Module):
  def __init__(self, vacob_size, emb_dim):
    super().__init__()
    self.embedding_matrix = nn.Parameter(torch.randn((vacob_size, emb_dim), dtype=torch.float))

  def forwart(self, x):
    return F.embedding(x, self.embedding_matrix)
    # WRITE ME

class SequenceTaggingNet(nn.Module):
  def __init__(self, word_num, emb_dim, hid_dim):
      super().__init__()
      self.emb = nn.Embedding(word_num, emb_dim)  # nn.Embeddingの使用
      self.rnn = nn.RNN(emb_dim, hid_dim, 1, batch_first=True)  # nn.RNNの使用
      self.linear = nn.Linear(hid_dim, 1)

  def forward(self, x, len_seq_max=0, len_seq=None, init_state=None):
      h = self.emb(x)
      if len_seq_max > 0:
          h, _ = self.rnn(h[:, 0:len_seq_max, :], init_state)
      else:
          h, _ = self.rnn(h, init_state)
      h = h.transpose(0, 1)
      if len_seq is not None:
          # 系列が終わった時点での出力を取る必要があるので len_seq を元に集約する
          h = h[len_seq - 1, list(range(len(x))), :]
      else:
          h = h[-1]

      y = self.linear(h)  # WRITE ME

      return y

In [11]:
emb_dim = 100
hid_dim = 50
n_epochs = 20
device = 'cuda'

net = SequenceTaggingNet(word_num, emb_dim, hid_dim)
net.to(device)
optimizer = optim.Adam(net.parameters())

for epoch in range(n_epochs):
    losses_train = []
    losses_valid = []

    net.train()
    n_train = 0
    acc_train = 0
    for label, line, len_seq in train_dataloader:
      net.zero_grad()

      t = label.to(device) # テンソルをGPUに移動
      x = line.to(device) # ( batch, time )
      len_seq.to(device)

      h = net(x, torch.max(len_seq), len_seq)
      y = torch.sigmoid(h).squeeze()

      loss = -torch.mean(t*torch_log(y) + (1 - t)*torch_log(1 - y))  # WRITE ME

      loss.backward()  # 誤差の逆伝播

      optimizer.step()  # パラメータの更新

      losses_train.append(loss.tolist())

      n_train += t.size()[0]

    # Valid
    t_valid = []
    y_pred = []
    net.eval()
    for label, line, len_seq in valid_dataloader:
      t = label.to(device) # テンソルをGPUに移動
      x = line.to(device)
      len_seq.to(device)

      h = net(x, torch.max(len_seq), len_seq)
      y = torch.sigmoid(h).squeeze()

      loss = -torch.mean(t*torch_log(y) + (1 - t)*torch_log(1 - y))  # WRITE ME

      pred = y.round().squeeze()  # 0.5以上の値を持つ要素を正ラベルと予測する
        # WRITE ME

      t_valid.extend(t.tolist())
      y_pred.extend(pred.tolist())

      losses_valid.append(loss.tolist())

    print('EPOCH: {}, Train Loss: {:.3f}, Valid Loss: {:.3f}, Validation F1: {:.3f}'.format(
        epoch,
        np.mean(losses_train),
        np.mean(losses_valid),
        f1_score(t_valid, y_pred, average='macro')
    ))

EPOCH: 0, Train Loss: 0.680, Valid Loss: 0.651, Validation F1: 0.619
EPOCH: 1, Train Loss: 0.620, Valid Loss: 0.607, Validation F1: 0.687
EPOCH: 2, Train Loss: 0.619, Valid Loss: 0.608, Validation F1: 0.684
EPOCH: 3, Train Loss: 0.540, Valid Loss: 0.577, Validation F1: 0.725
EPOCH: 4, Train Loss: 0.559, Valid Loss: 0.597, Validation F1: 0.687
EPOCH: 5, Train Loss: 0.568, Valid Loss: 0.636, Validation F1: 0.639
EPOCH: 6, Train Loss: 0.535, Valid Loss: 0.605, Validation F1: 0.686
EPOCH: 7, Train Loss: 0.457, Valid Loss: 0.580, Validation F1: 0.718
EPOCH: 8, Train Loss: 0.422, Valid Loss: 0.563, Validation F1: 0.749
EPOCH: 9, Train Loss: 0.458, Valid Loss: 0.625, Validation F1: 0.688
EPOCH: 10, Train Loss: 0.391, Valid Loss: 0.565, Validation F1: 0.754
EPOCH: 11, Train Loss: 0.336, Valid Loss: 0.582, Validation F1: 0.745
EPOCH: 12, Train Loss: 0.412, Valid Loss: 0.590, Validation F1: 0.702
EPOCH: 13, Train Loss: 0.406, Valid Loss: 0.593, Validation F1: 0.738
EPOCH: 14, Train Loss: 0.328, 

In [ ]:
net.eval()

y_pred = []
for _, line, len_seq in test_dataloader:

    x = line.to(device)
    len_seq.to(device)

    h = net(x, torch.max(len_seq), len_seq)
    y = torch.sigmoid(h).squeeze()

    pred = y.round().squeeze()  # 0.5以上の値を持つ要素を正ラベルと予測する

    y_pred.extend(pred.tolist())


submission = pd.Series(y_pred, name='label')
submission.to_csv(work_dir + '/Lecture07/submission_pred_07.csv', header=True, index_label='id')